In [5]:
from collections import defaultdict
from pathlib import Path
import re
from ase.io.trajectory import Trajectory
from readqe import read_qe_hubbard_out

outputs_dir = Path("outputs")
traj_dir = outputs_dir / "traj"
traj_dir.mkdir(parents=True, exist_ok=True)

u_file_pattern = re.compile(r"^(?P<system>.+)_combi_u(?P<u>\d+)\.out$")

groups = defaultdict(list)
for out_file in outputs_dir.glob("*_combi_u*.out"):
    m = u_file_pattern.match(out_file.name)
    if m:
        groups[m.group("system")].append((int(m.group("u")), out_file))

if not groups:
    raise FileNotFoundError(f"No matching *_combi_u*.out files found in {outputs_dir}")

written = 0
for system in sorted(groups):
    ordered_files = sorted(groups[system], key=lambda x: x[0])
    traj_path = traj_dir / f"{system}_combi.traj"

    total_frames = 0
    with Trajectory(str(traj_path), "w") as t:
        for u_value, out_file in ordered_files:
            snapshots = read_qe_hubbard_out(str(out_file), index=":")

            if not isinstance(snapshots, list):
                snapshots = [snapshots]

            if not snapshots:
                print(f"Skipping empty snapshots in {out_file.name}")
                continue

            for i, atoms in enumerate(snapshots):
                frame = atoms.copy()
                frame.info["source_out"] = out_file.name
                frame.info["u_value"] = u_value
                frame.info["frame_index_in_out"] = i
                frame.info["global_frame_index"] = total_frames
                t.write(frame)
                total_frames += 1

    written += 1
    print(f"Wrote {total_frames} frames from {len(ordered_files)} .out files -> {traj_path}")

print(f"Done: wrote {written} combined .traj files in {traj_dir}")

Wrote 15 frames from 1 .out files -> outputs/traj/TiN_Li2S_combi.traj
Wrote 19 frames from 1 .out files -> outputs/traj/TiN_Li2S2_combi.traj
Wrote 112 frames from 3 .out files -> outputs/traj/TiN_Li2S4_combi.traj
Wrote 200 frames from 5 .out files -> outputs/traj/TiN_Li2S6_combi.traj
Wrote 157 frames from 4 .out files -> outputs/traj/TiN_Li2S8_combi.traj
Wrote 176 frames from 3 .out files -> outputs/traj/TiN_S8_combi.traj
Wrote 30 frames from 1 .out files -> outputs/traj/VN_Li2S_combi.traj
Wrote 34 frames from 1 .out files -> outputs/traj/VN_Li2S2_combi.traj
Wrote 33 frames from 1 .out files -> outputs/traj/VN_Li2S4_combi.traj
Wrote 161 frames from 7 .out files -> outputs/traj/VN_Li2S6_combi.traj
Wrote 164 frames from 6 .out files -> outputs/traj/VN_Li2S8_combi.traj
Wrote 156 frames from 7 .out files -> outputs/traj/VN_S8_combi.traj
Done: wrote 12 combined .traj files in outputs/traj


In [7]:
from ase.io import read
from ase.visualize import view

surfaces = ["TiN", "VN"]
adsorbates = ["Li2S8"]

for surface in surfaces:
    for ads in adsorbates:
        traj_path = traj_dir / f"{surface}_{ads}_combi.traj"

        if not traj_path.exists():
            print(f"No combined traj found for {surface}-{ads}: {traj_path.name}")
            continue

        frames = read(str(traj_path), index=":")
        if not isinstance(frames, list):
            frames = [frames]

        print(f"Viewing {traj_path.name} ({len(frames)} frames)")
        view(frames)

Viewing TiN_Li2S8_combi.traj (157 frames)
Viewing VN_Li2S8_combi.traj (164 frames)


In [3]:
from collections import defaultdict
from pathlib import Path
import re
import numpy as np
from ase import Atoms
from ase.io import read, write
from readqe import read_qe_hubbard_out

RY_TO_EV = 13.605693009
BOHR_TO_ANG = 0.529177210903

outputs_dir = Path("outputs")
final_xyz_dir = Path("final_xyz")
final_xyz_dir.mkdir(parents=True, exist_ok=True)

u_file_pattern = re.compile(r"^(?P<system>.+)_combi_u(?P<u>\d+)\.out$")
energy_pattern = re.compile(
    r"^\s*!?\s*total energy\s*=\s*([-+]?\d*\.?\d+(?:[eEdD][-+]?\d+)?)\s*Ry",
    re.MULTILINE,
)

def last_total_energy_ev(out_path: Path):
    text = out_path.read_text(errors="ignore")
    matches = energy_pattern.findall(text)
    if not matches:
        return None

    last_ry = float(matches[-1].replace("D", "E").replace("d", "e"))
    return last_ry * RY_TO_EV

def parse_qe_final_coordinates(out_path: Path):
    text = out_path.read_text(errors="ignore")

    final_block_match = re.search(
        r"Begin final coordinates(.*?)End final coordinates",
        text,
        flags=re.DOTALL,
    )
    if not final_block_match:
        raise ValueError("No 'Begin final coordinates' block found")

    final_block = final_block_match.group(1)
    apos_match = re.search(
        r"ATOMIC_POSITIONS\s*\(([^)]+)\)\s*\n(.*)",
        final_block,
        flags=re.DOTALL,
    )
    if not apos_match:
        raise ValueError("No ATOMIC_POSITIONS block found in final coordinates")

    positions_unit = apos_match.group(1).strip().lower()
    coords_text = apos_match.group(2).strip()

    symbols = []
    positions = []
    for line in coords_text.splitlines():
        parts = line.split()
        if len(parts) < 4:
            continue
        if parts[0].upper().startswith("END") or parts[0].startswith("!"):
            continue
        try:
            x, y, z = map(float, parts[1:4])
        except ValueError:
            continue
        symbols.append(parts[0])
        positions.append([x, y, z])

    if not symbols:
        raise ValueError("No atomic coordinates parsed from final coordinates block")

    positions = np.array(positions, dtype=float)
    if positions_unit in {"bohr", "a.u.", "au"}:
        positions *= BOHR_TO_ANG

    alat_match = re.search(r"lattice parameter \(alat\)\s*=\s*([0-9.EeDd+-]+)", text)
    axes_match = re.search(
        r"crystal axes:\s*\(cart\. coord\. in units of alat\)\s*\n"
        r"\s*a\(1\) = \(\s*([-.0-9EeDd+]+)\s+([-.0-9EeDd+]+)\s+([-.0-9EeDd+]+)\s*\)\s*\n"
        r"\s*a\(2\) = \(\s*([-.0-9EeDd+]+)\s+([-.0-9EeDd+]+)\s+([-.0-9EeDd+]+)\s*\)\s*\n"
        r"\s*a\(3\) = \(\s*([-.0-9EeDd+]+)\s+([-.0-9EeDd+]+)\s+([-.0-9EeDd+]+)\s*\)",
        text,
    )

    cell = None
    if alat_match and axes_match:
        alat_bohr = float(alat_match.group(1).replace("D", "E").replace("d", "e"))
        a1 = [float(axes_match.group(i).replace("D", "E").replace("d", "e")) for i in (1, 2, 3)]
        a2 = [float(axes_match.group(i).replace("D", "E").replace("d", "e")) for i in (4, 5, 6)]
        a3 = [float(axes_match.group(i).replace("D", "E").replace("d", "e")) for i in (7, 8, 9)]
        cell = np.array([a1, a2, a3], dtype=float) * alat_bohr * BOHR_TO_ANG

    if cell is None:
        atoms = Atoms(symbols=symbols, positions=positions)
        atoms.pbc = False
    else:
        atoms = Atoms(symbols=symbols, positions=positions, cell=cell, pbc=True)

    return atoms

groups = defaultdict(list)
for out_file in outputs_dir.glob("*_combi_u*.out"):
    m = u_file_pattern.match(out_file.name)
    if m:
        groups[m.group("system")].append((int(m.group("u")), out_file))

if not groups:
    raise FileNotFoundError(f"No matching *_combi_u*.out files found in {outputs_dir}")

written_combi = 0
for system in sorted(groups):
    final_u, final_out = max(groups[system], key=lambda x: x[0])
    snapshots = read_qe_hubbard_out(str(final_out), index=":")

    if not isinstance(snapshots, list):
        snapshots = [snapshots]

    if not snapshots:
        print(f"Skipping {final_out.name}: no geometry snapshots")
        continue

    final_atoms = snapshots[-1].copy()
    energy_ev = last_total_energy_ev(final_out)

    if energy_ev is not None:
        final_atoms.info["energy"] = energy_ev
        final_atoms.info["energy_eV"] = energy_ev

    final_atoms.info["source_out"] = final_out.name
    final_atoms.info["u_value"] = final_u

    xyz_path = final_xyz_dir / f"{system}_final.xyz"
    write(str(xyz_path), final_atoms, format="extxyz")
    written_combi += 1

    if energy_ev is None:
        print(f"Wrote {xyz_path} from {final_out.name} (energy not found)")
    else:
        print(f"Wrote {xyz_path} from {final_out.name} (E = {energy_ev:.8f} eV)")

slab_outputs_dir = Path("../slab/output_files")
written_slab = 0
for slab_out in sorted(slab_outputs_dir.glob("*_slab.out")):
    try:
        slab_atoms = read(str(slab_out), format="espresso-out", index=-1)
    except Exception:
        try:
            slab_atoms = parse_qe_final_coordinates(slab_out)
            print(f"Fallback parser used for {slab_out.name}")
        except Exception as e:
            print(f"Skipping {slab_out}: could not parse final structure ({e})")
            continue

    slab_atoms = slab_atoms.copy()
    slab_atoms.info["source_out"] = slab_out.name
    slab_atoms.info["system"] = slab_out.stem.replace("_slab", "")
    slab_atoms.info["calculation"] = "slab"

    slab_xyz_path = final_xyz_dir / f"{slab_out.stem}_final.xyz"
    write(str(slab_xyz_path), slab_atoms, format="extxyz")
    written_slab += 1
    print(f"Wrote {slab_xyz_path} from {slab_out.name}")

print(f"Done: wrote {written_combi} combi final xyz files in {final_xyz_dir}")
print(f"Done: wrote {written_slab} slab final xyz files in {final_xyz_dir}")

Wrote final_xyz/TiN_Li2S_final.xyz from TiN_Li2S_combi_u1.out (E = -137194.29316326 eV)
Wrote final_xyz/TiN_Li2S2_final.xyz from TiN_Li2S2_combi_u1.out (E = -137525.03254084 eV)
Wrote final_xyz/TiN_Li2S4_final.xyz from TiN_Li2S4_combi_u3.out (E = -138181.52477987 eV)
Wrote final_xyz/TiN_Li2S6_final.xyz from TiN_Li2S6_combi_u5.out (E = -138838.79891590 eV)
Wrote final_xyz/TiN_Li2S8_final.xyz from TiN_Li2S8_combi_u4.out (E = -139495.50413437 eV)
Wrote final_xyz/TiN_S8_final.xyz from TiN_S8_combi_u3.out (E = -139101.69981426 eV)
Wrote final_xyz/VN_Li2S_final.xyz from VN_Li2S_combi_u1.out (E = -162527.17329191 eV)
Wrote final_xyz/VN_Li2S2_final.xyz from VN_Li2S2_combi_u1.out (E = -162856.44070195 eV)
Wrote final_xyz/VN_Li2S4_final.xyz from VN_Li2S4_combi_u1.out (E = -163512.93623818 eV)
Wrote final_xyz/VN_Li2S6_final.xyz from VN_Li2S6_combi_u7.out (E = -164173.11232186 eV)
Wrote final_xyz/VN_Li2S8_final.xyz from VN_Li2S8_combi_u6.out (E = -164826.85468516 eV)
Wrote final_xyz/VN_S8_final.xy

In [5]:
slab = read_qe_hubbard_out('/home/ameer_ubuntu/Git_projects/QE_2/u/slab/output_files/VN_slab.out')